# Estimation d'un système de demande d'énergie résidentielle par régression apparemment non reliée


## Synthèse

Un service public régional estime un **système de demande d'énergie** à deux équations pour le résidentiel — les parts budgétaires de l'électricité et du gaz naturel en fonction du prix propre, du prix croisé et du revenu du ménage — à l'aide de **PROC SYSLIN** avec la méthode de régression apparemment non reliée (SUR). Les deux équations de parts partagent un budget de ménage commun, si bien que leurs erreurs sont corrélées entre elles ; la SUR estime les équations conjointement pour exploiter cette corrélation, retrouvant des effets de prix propre et croisé cohérents en signe et fournissant la covariance inter-équations et les tests de restriction dont un analyste de la demande a besoin.


## Sources de données

| Jeu de données | Lignes | Granularité | Variables clés | Description |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | une ligne par observation mensuelle de marché | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Panel synthétique mensuel du marché de l'énergie résidentielle. `lp_elec`/`lp_gas` sont les log-prix réels de l'électricité et du gaz naturel ; `lincome` est le log du revenu réel du ménage ; `w_elec`/`w_gas` sont les parts budgétaires de dépenses pour l'électricité et le gaz naturel, générées à partir d'une structure de demande connue de type AIDS plus un bruit inter-équations corrélé. |


# Estimation d'un système de demande d'énergie résidentielle par régression apparemment non reliée

Un service public régional de gaz et d'électricité veut comprendre comment les ménages réallouent leur budget énergétique entre l'**électricité** et le **gaz naturel** lorsque les prix relatifs et le revenu changent. Le cadre naturel est un **système de demande** : un ensemble d'équations de parts budgétaires estimées conjointement.

Deux caractéristiques font de l'estimation conjointe le bon outil :

- Les équations de parts puisent dans un budget de ménage commun, si bien que leurs erreurs sont **corrélées entre elles** — lorsqu'un ménage dépense plus en électricité, il dépense moins en gaz. Estimer les équations ensemble par **régression apparemment non reliée (SUR)** utilise cette corrélation pour gagner en efficacité.
- La théorie économique impose des **restrictions linéaires** (sommation, homogénéité, symétrie) qu'un estimateur de système peut faire respecter directement.

`PROC SYSLIN` avec l'option `SUR` est conçu exactement pour cette situation. Il ajuste chaque équation de parts, estime la covariance des erreurs inter-équations, et réajuste le système avec cette covariance pour livrer des élasticités efficaces et cohérentes avec la théorie — de pair avec la matrice de covariance inter-modèles et des tests de restriction conjoints.

Dans ce notebook, nous :

1. Générons un panel synthétique mensuel du marché de l'énergie à partir d'une structure de parts connue.
2. Ajustons un système de parts à deux équations par SUR.
3. Comparons l'ajustement conjoint SUR avec les MCO équation par équation.
4. Imposons une restriction d'homogénéité et lisons son test F conjoint.
5. Extrayons la table de coefficients pour le rapport d'élasticités.


## Étape 1 — Générer un panel synthétique du marché de l'énergie

Nous simulons 60 observations mensuelles d'un petit **système de demande d'énergie à deux biens** (électricité et gaz naturel). Le processus générateur des données est un modèle de parts budgétaires linéarisé, de type AIDS :

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

Nous intégrons délibérément une **corrélation inter-équations** : lorsque les ménages dépensent plus en électricité, ils dépensent moins en gaz, si bien que `e1` et `e2` sont négativement corrélés. Un facteur de prix commun du marché de l'énergie entraîne aussi les deux log-prix ensemble. Ce sont les caractéristiques qui récompensent l'estimation conjointe SUR par rapport à l'ajustement de chaque équation isolément.


In [1]:
DONNÉES energy;
   APPELER streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   FAIRE month = 1 JUSQU_À 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      SORTIE;
   FIN;

   GARDER month lp_elec lp_gas lincome w_elec w_gas;
EXÉCUTER;

PROCÉDURE MOYENNES DONNÉES=energy n mean std MIN MAX maxdec=3;
   VAR w_elec w_gas lp_elec lp_gas lincome;
EXÉCUTER;

                                                  The MEANS Procedure

 Variable        N           Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------
 W_ELEC         60          0.440       0.011       0.417       0.464
 W_GAS          60          0.228       0.014       0.198       0.256
 LP_ELEC        60          4.617       0.142       4.354       4.911
 LP_GAS         60          4.211       0.162       3.818       4.566
 LINCOME        60         10.916       0.088      10.723      11.174
 --------------------------------------------------------------------



NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Étape 2 — Estimation SUR de référence du système de demande

Nous estimons les deux équations de parts conjointement. L'option `SUR` indique à `PROC SYSLIN` d'estimer la covariance des erreurs inter-équations et de l'utiliser pour un ajustement efficace par MCG réalisables. Deux instructions `MODEL` étiquetées (`elec` et `gas`) définissent le système ; chacune régresse une part budgétaire sur les deux log-prix et le log-revenu.

SYSLIN rapporte les estimations de paramètres de chaque équation et, à la fin, la **matrice de covariance inter-modèles** — la covariance estimée des erreurs des deux équations que la SUR exploite.


In [2]:
PROCÉDURE syslin DONNÉES=energy sur;
   elec: MODÈLE w_elec = lp_elec lp_gas lincome;
   gas:  MODÈLE w_gas  = lp_elec lp_gas lincome;
EXÉCUTER;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Étape 3 — Comparer avec les MCO équation par équation

Pour voir ce que la SUR nous apporte, nous réajustons les deux mêmes équations par les moindres carrés ordinaires (la méthode par défaut, une équation à la fois). Les MCO ignorent la corrélation des erreurs inter-équations, ils sont donc convergents mais moins efficaces que la SUR lorsque cette corrélation est non nulle — comme c'est le cas ici par construction.

Comparer les deux tables de coefficients montre comment les estimations et leurs écarts-types se déplacent une fois la structure du système prise en compte.


In [3]:
PROCÉDURE syslin DONNÉES=energy ols;
   elec: MODÈLE w_elec = lp_elec lp_gas lincome;
   gas:  MODÈLE w_gas  = lp_elec lp_gas lincome;
EXÉCUTER;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Étape 4 — Imposer la théorie économique et la tester

La théorie de la demande dit que les effets nominaux prix/revenu doivent obéir à l'**homogénéité de degré zéro** : mettre à l'échelle tous les prix et le revenu laisse la demande réelle inchangée, si bien que les coefficients de prix et de revenu au sein d'une équation somment à zéro. Nous imposons cela sur la part d'électricité avec une instruction `RESTRICT`.

SYSLIN réajuste le système sous la contrainte et rapporte un test F de **Restriction Results** indiquant si la restriction est cohérente avec les données — un test direct et conjoint de l'hypothèse d'homogénéité.


In [4]:
PROCÉDURE syslin DONNÉES=energy sur;
   elec: MODÈLE w_elec = lp_elec lp_gas lincome;
   gas:  MODÈLE w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
EXÉCUTER;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Étape 5 — Capturer les estimations pour le rapport d'élasticités

Pour un rapport final, nous conservons les coefficients convergés avec `OUTEST=`. Le jeu de données résultant porte une ligne par équation avec les estimations d'ordonnée à l'origine et de pente ainsi que les statistiques d'ajustement, qui alimentent les calculs d'élasticités de prix propre et croisé du service public aux parts moyennes de l'échantillon. Nous l'imprimons pour archive.


In [5]:
PROCÉDURE syslin DONNÉES=energy sur outest=demand_est;
   elec: MODÈLE w_elec = lp_elec lp_gas lincome;
   gas:  MODÈLE w_gas  = lp_elec lp_gas lincome;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=demand_est noobs;
   TITRE "SUR demand-system coefficient estimates";
EXÉCUTER;
TITRE;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## Interprétation des résultats

**Cohérence des signes.** Les estimations SUR retrouvent la structure de substitution intégrée aux données. Dans l'équation de la part d'électricité, le **coefficient du log-prix propre est positif** (`LP_ELEC` = 0,112) tandis que le **coefficient de prix croisé est négatif** (`LP_GAS` = -0,098) ; dans l'équation de la part de gaz, le schéma est symétrique (propre `LP_GAS` = 0,114, croisé `LP_ELEC` = -0,075). Chaque effet de prix propre et croisé est statistiquement significatif (chaque `Pr > |t|` inférieur à 0,005), si bien que les signes de substitution sont fermement identifiés plutôt que des artefacts d'un ajustement bruité.

**La SUR exploite la corrélation inter-équations.** La **matrice de covariance inter-modèles** montre un terme hors diagonale négatif (-0,000068) : les ménages arbitrent la dépense d'électricité contre la dépense de gaz, exactement comme construit. Comme cette covariance est non nulle, l'estimation conjointe SUR est plus efficace que l'ajustement de chaque équation isolément — les écarts-types SUR de l'Étape 2 sont uniformément un peu plus petits que les écarts-types des MCO équation par équation de l'Étape 3 (par exemple, l'écart-type de `LP_ELEC` pour le gaz chute de 0,0264 sous MCO à 0,0255 sous SUR), tandis que les estimations ponctuelles restent inchangées. Cette inférence plus resserrée est le bénéfice de modéliser le système plutôt que deux régressions séparées.

**La théorie tient.** Imposer l'**homogénéité de degré zéro** sur la part d'électricité (les coefficients de prix et de revenu sommant à zéro) via `RESTRICT` a produit un test F de Restriction Results de 2,14 avec `Pr > F` = 0,149. La restriction n'est **pas rejetée**, si bien que la prédiction d'homogénéité de la théorie de la demande du consommateur est cohérente avec cet échantillon — le service public peut porter les estimations contraintes et cohérentes avec la théorie dans le rapport en toute confiance.

**Usage opérationnel.** La table `OUTEST=` conserve une ligne par équation avec les estimations d'ordonnée à l'origine et de pente et les statistiques d'ajustement. Ces coefficients se convertissent directement en élasticités de prix propre et croisé et en une élasticité de revenu (de dépense) aux parts moyennes de l'échantillon (`W_ELEC` ~ 0,44, `W_GAS` ~ 0,23 de l'Étape 1), donnant au service public une base défendable et cohérente avec la théorie pour la prévision de la demande et les scénarios de conception tarifaire.
